# LunarLander DQN vs PPO: runs on Kaggle

Runs part of the training jobs of [amine-belkacem/lunarlander-dqn-vs-ppo](https://github.com/amine-belkacem/lunarlander-dqn-vs-ppo) on a Kaggle CPU machine.

Before running: **Settings > Accelerator: None** (CPU only, a GPU does not help these small networks) and **Settings > Internet: On** (needed to clone the repository).

1. First time on this machine: run the cells up to the speed test, one by one, and note the measured speed.
2. Then set `RUN_SPEED_TEST = False` and use **Save Version > Save & Run All (Commit)**. It runs in the background (12 h max) and the results end up in the notebook output as `results.zip`.

In [ ]:
# The only cell to edit.
JOBS = ["dqn:3-8"]           # notebook A. Notebook B: ["dqn:9", "dqn_big_buffer:0-4"]
RUN_SPEED_TEST = True        # 3 short DQN runs in parallel (a few minutes) to measure this machine
MAX_PARALLEL = 3
COMMIT = "2b4367ac574a2ebceba653ce28bed87a6b8bd45f"          # exact code version used for every run

In [ ]:
import os, platform, sys
print("python", platform.python_version())
print("CPUs  ", os.cpu_count())
with open("/proc/meminfo") as f:
    print("RAM   ", f.readline().split()[1:3])
with open("/proc/cpuinfo") as f:
    print("model ", next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name")))

In [ ]:
REPO = "/tmp/lunarlander-dqn-vs-ppo"
!rm -rf {REPO}
!git clone -q https://github.com/amine-belkacem/lunarlander-dqn-vs-ppo.git {REPO}
!cd {REPO} && git checkout -q {COMMIT} && git log --oneline -1
# CPU-only torch wheel, same version as on my laptop and in CI.
# --no-warn-conflicts: Kaggle's preinstalled packages (torchvision,
# kaggle-environments...) expect other versions; they are not used here.
!pip install -q --no-warn-conflicts torch==2.14.1 --index-url https://download.pytorch.org/whl/cpu
!pip install -q --no-warn-conflicts -r {REPO}/requirements.txt
!pip install -q --no-warn-conflicts -e {REPO} --no-deps
!cd {REPO} && PYTHONPATH={REPO}/src python -m pytest -q -m "not slow" 2>&1 | tail -n 2

In [ ]:
import glob, json, math
# The editable install above is only seen by new Python processes; this
# kernel started before it, so the source folder is added by hand.
sys.path.insert(0, f"{REPO}/src")
from lunarlander_rl.launcher import parse_job_spec

LAPTOP_STEPS_PER_S = 139  # 3 DQN runs in parallel on my laptop (Ryzen 3 7320U)

if RUN_SPEED_TEST:
    !rm -rf /tmp/speed
    !cd {REPO} && PYTHONPATH={REPO}/src python -m lunarlander_rl.launcher dqn:100-102 --runs-dir /tmp/speed --total-timesteps 10000 --stagger-seconds 5 --max-parallel {MAX_PARALLEL}
    metas = [json.load(open(p)) for p in glob.glob("/tmp/speed/dqn/seed_*/meta.json")]
    rates = [10_000 / m["wall_seconds_train"] for m in metas if m.get("status") == "done"]
    if not rates:
        !cat /tmp/speed/logs/*.log
        raise RuntimeError("speed test produced no finished run, see the logs above")
    slowest = min(rates)
    hours_per_run = 1_000_000 / slowest / 3600
    jobs = [job for spec in JOBS for job in parse_job_spec(spec)]
    n_dqn = sum(job.config.startswith("dqn") for job in jobs)
    total = math.ceil(n_dqn / MAX_PARALLEL) * hours_per_run
    print(f"DQN speed with {MAX_PARALLEL} runs in parallel: {', '.join(f'{r:.0f}' for r in rates)} steps/s "
          f"(laptop: {LAPTOP_STEPS_PER_S})")
    print(f"-> about {hours_per_run:.1f} h per 1M-step DQN run, about {total:.1f} h for the {n_dqn} DQN runs of JOBS")
    if total > 11:
        print("WARNING: too close to the 12 h limit, split JOBS over more notebooks")

In [ ]:
RUNS = "/kaggle/working/runs"
!cd {REPO} && PYTHONPATH={REPO}/src python -m lunarlander_rl.launcher {" ".join(JOBS)} --runs-dir {RUNS} --max-parallel {MAX_PARALLEL}

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/results", "zip", RUNS)
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")